# Phase 3 training on Google Colab (actions A2 and A8)

The laptop runs YOLOv8n (seed 0) and YOLO11n for the three categories. This notebook runs the rest on
Colab's free GPU, one model per run:

| Run | `MODEL` | `SEED` |
|---|---|---|
| Non-YOLO baseline (A8, needed) | `rtdetr-l` | 0 |
| Larger YOLO baseline (A8) | `yolov8s` | 0 |
| Seed variation of the proposed model (A2, optional) | `yolov8n` | 1 and 2 |

Everything is downloaded to this Colab machine's temporary disk; **nothing is saved to Google Drive**.
At the end you download one small zip (results plus the trained weights: 6 to 70 MB) to your computer.

Steps: `Runtime > Change runtime type > T4 GPU`, set the three values below, then `Runtime > Run all`.
If Colab disconnects during training, run all cells again: training resumes from the last epoch.

In [7]:
CATEGORY = "pothole"   # "pothole", "garbage_litter" or "graffiti"
MODEL = "rtdetr-l"     # "rtdetr-l", "yolov8s" or "yolov8n"
SEED = 0
BATCH = 8 if MODEL.startswith("rtdetr") else 16   # RT-DETR-l does not fit batch 16 on a T4

In [8]:
!nvidia-smi -L
import os
if not os.path.exists("/content/urbanfix"):
    !git clone --depth 1 https://github.com/pizzaparth/urbanfix.git /content/urbanfix
%cd /content/urbanfix/ai_model
!pip -q install ultralytics==8.4.173

GPU 0: Tesla T4 (UUID: GPU-f37f417c-17e6-198e-a7b8-fa038ba892cb)
/content/urbanfix/ai_model


## Data (from the public sources, onto this machine's disk)

In [9]:
if CATEGORY == "pothole":
    !python scripts/fetch_rdd2022.py --countries India Japan Czech United_States China_MotorBike
elif CATEGORY == "garbage_litter":
    !python scripts/prepare_sources.py --only taco
    !python scripts/fetch_taco_unofficial.py
else:
    !python scripts/prepare_sources.py --only storm
!python scripts/fetch_coco_negatives.py
!python scripts/build_yolo_sets.py {CATEGORY}

India.zip: 527 / 527 MB
India: 7706 training images
Japan.zip: 1073 / 1073 MB
Japan: 10506 training images
Czech.zip: 257 / 257 MB
Czech: 2829 training images
United_States.zip: 444 / 444 MB
United_States: 4805 training images
China_MotorBike.zip: 192 / 192 MB
China_MotorBike: 1977 training images
1000 of 1000 images on disk
pothole {'train': {'images': 1600, 'images_with_defect': 1281, 'objects': 1959}, 'val': {'images': 341, 'images_with_defect': 275, 'objects': 419}, 'test': {'images': 339, 'images_with_defect': 268, 'objects': 432}}
pothole_rdd2022_india {'images': 1911, 'images_with_defect': 1530, 'objects': 3187}
pothole_roboflow: not on this machine, skipped


## Train (40 epochs at most, early stopping after 10; checkpoint chosen on the validation split only)

In [10]:
!python scripts/train.py {CATEGORY} --model {MODEL} --seed {SEED} --batch {BATCH} --device 0 --resume

New https://pypi.org/project/ultralytics/8.4.219 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.173 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/urbanfix/ai_model/data/v2/pothole/data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, mod

## Evaluate once on the test and external sets, then score images for the complaint-level analysis

In [ ]:
TAG = f"{CATEGORY}__{MODEL}__s{SEED}"
!python scripts/evaluate.py --tags {TAG} --device 0
!python scripts/score_images.py --tags {TAG} --device 0
!python scripts/complaint_eval.py --tag {TAG}

Ultralytics 8.4.173 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary (fused): 315 layers, 31,985,795 parameters, 0 gradients, 105.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 32.7±24.2 MB/s, size: 145.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/urbanfix/ai_model/data/v2/pothole/labels/test... 339 images, 71 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 339/339 433.3it/s 0.8s
val: New cache created: /content/urbanfix/ai_model/data/v2/pothole/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 1.5it/s 14.6s
                   all        339        432      0.701      0.607      0.671      0.311
Speed: 0.9ms preprocess, 39.8ms inference, 0.0ms loss, 0.2ms postprocess per image
Ultralytics 8.4.173 🚀 Python-3.13.16 torch-2.14.1+cu13

## Download the results to your computer

Put the zip in `ai_model/` of your local repository and unzip it there (it adds files under `results/` and `weights/v2/`). The CPU latency is then remeasured on the laptop with `python scripts/evaluate.py --latency-only --tags <tag>`, so all models are timed on the same CPU.

In [ ]:
import shutil, os
out = f"/content/{TAG}"
for sub in ("train", "eval", "scores", "complaints"):
    os.makedirs(f"{out}/results/{sub}", exist_ok=True)
    src = f"results/{sub}/{TAG}.json"
    if os.path.exists(src):
        shutil.copy(src, f"{out}/results/{sub}/")
os.makedirs(f"{out}/weights/v2", exist_ok=True)
shutil.copy(f"weights/v2/{TAG}.pt", f"{out}/weights/v2/")
shutil.make_archive(out, "zip", out)
from google.colab import files
files.download(f"{out}.zip")